[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peterropac/Aegolius/blob/main/examples/LLM/claude_sonnet_5_couch_from_guide.ipynb)

In [ ]:
#@title Setup
import importlib.util
if importlib.util.find_spec('google.colab') is not None:
    %pip install -q spomso


## Setup

**Test date: 31.7.2026**

**Claude Sonnet 5 (Medium)** was tasked with generating a **modern couch** using the **SPOMSO** package. Claude was provided with the guide for SPOMSO (1.5.0), the .zip file of the package, and the following prompt:

*Read through this zip file. It contains the Python package SPOMSO. I want you to design a modern couch using this Python package. As the output I want some code that I would run and it would output the point cloud of the couch and some figures. As help you also get a guide in form of an .md file. Everything else I leave up to you. Good luck and if you have any questions feel free to ask.*

## Response:

This is actually version 1.5.0 — matching the guide exactly. Let me install this directly from source instead of the PyPI 1.4.0.Now installed properly from source at 1.5.0, matching the guide. Let's verify with a quick sanity test.Everything works with the actual 1.5.0 package. Now let's design the couch. I'll build a modern two-seat sofa: smoothly blended base cushion, two rolled seat cushions, two rolled back cushions, angled rolled arms, and slim tapered wooden legs — all using smooth unions for that soft upholstered look.Good, the basic shape works. Now let's iterate visually — render cross-sections quickly to check proportions before finalizing.This looks like a solid, clean modern couch silhouette — tapered legs, low frame, seat + back cushions, flat-top arms. Let's check a 3D scatter of the surface points to make sure it reads well in 3D before finalizing the full script.Fast enough. Now let's write the final, polished, well-commented script.Let's check the resulting figures before delivering.This looks great — clear part-colored couch (walnut legs, dark frame, beige seat/back cushions, warm grey arms), clean cross-sections, and a proper surface point cloud. Let's copy everything to outputs and hand it over.I noticed your zip had the actual 1.5.0 source (matching your guide exactly, ahead of what's on PyPI), so I installed from that directly rather than the older PyPI release.

The design: a two-seat modern sofa built from SDF primitives —

- **4 tapered walnut legs** (cones, wide where they meet the frame, narrow at the floor)
- **A low frame** the whole thing sits on
- **2 seat cushions** with a visible seam between them
- **2 backrest cushions**, tilted back ~9°
- **2 flat-top track arms**, smoothly blended into the frame (everything else uses sharp unions so cushion seams stay visible)

Running `modern_couch.py` gives you:
- `couch_surface_points.npy` / `.xyz` — the surface point cloud (~116k points)
- `couch_cross_sections.png` — side/front/top binary slices
- `couch_pointcloud_3d.png` — two 3D scatter views, color-coded by part (I evaluate each part's own SDF at the surface points and tag each point by whichever part it's deepest inside)

All dimensions are named constants at the top of the relevant sections, so you can easily change proportions (e.g. `ARM_HEIGHT`, `BACK_TILT`, `CO_RESOLUTION` for point density) and rerun.

In [1]:
"""
Modern Couch — procedural SDF design with SPOMSO
==================================================

Builds a two-seat modern sofa out of signed-distance-function primitives:

    - 4 tapered wooden legs (Cone)
    - a low supporting frame (Box)
    - 2 seat cushions with a visible seam between them (Box)
    - 2 backrest cushions, tilted back slightly (Box)
    - 2 flat-top track arms (Box)

Sharp unions are used where a part visibly rests on / is attached to
another (cushions on the frame, legs under the frame), and smooth unions
are used only where the arms meet the frame, so the arm reads as a single
continuous upholstered piece rather than two boxes glued together.

Running this script produces:
    - couch_surface_points.npy   : (N, 3) surface point cloud
    - couch_surface_points.xyz   : same data, plain-text xyz format
    - couch_cross_sections.png   : side / front / top binary cross-sections
    - couch_pointcloud_3d.png    : two 3D scatter views, colour-coded by part

Requires: spomso, numpy, matplotlib
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3D projection)

from spomso.cores import generate_grid, smarter_reshape, CombineGeometry
from spomso.cores.geom_3d import Box, Cone
from spomso.cores.post_processing import hard_binarization


# ----------------------------------------------------------------------------
# 1. Grid
# ----------------------------------------------------------------------------
# Physical size of the design domain (metres): width x, depth y, height z.
CO_SIZE = (2.10, 1.05, 1.00)
CO_RESOLUTION = (180, 90, 85)

coor, co_res = generate_grid(CO_SIZE, CO_RESOLUTION)
FLOOR = -CO_SIZE[2] / 2  # generate_grid always centres the domain at the origin

UNION = CombineGeometry("UNION")
SMOOTH_UNION = CombineGeometry("SMOOTH_UNION2")


def smooth(a, b, width=0.05):
    return SMOOTH_UNION.combine_parametric(a, b, parameters=width)


# ----------------------------------------------------------------------------
# 2. Legs — tapered cones, wide where they meet the frame, narrow at the floor
# ----------------------------------------------------------------------------
LEG_HEIGHT = 0.16
LEG_ANGLE = np.deg2rad(5.5)
FRAME_HEIGHT = 0.20
FRAME_BOTTOM = FLOOR + LEG_HEIGHT

leg_xy_positions = [(0.84, 0.34), (0.84, -0.34), (-0.84, 0.34), (-0.84, -0.34)]
legs = []
for x, y in leg_xy_positions:
    leg = Cone(LEG_HEIGHT, LEG_ANGLE)
    leg.rotate(np.deg2rad(180), (1, 0, 0))     # flip so the point faces the floor
    leg.move((x, y, FLOOR + leg.height_offset))
    legs.append(leg)

# ----------------------------------------------------------------------------
# 3. Frame — the low box the cushions and arms sit on
# ----------------------------------------------------------------------------
frame = Box(1.86, 0.92, FRAME_HEIGHT)
frame.rounding(0.035)
frame.move((0, 0, FRAME_BOTTOM + FRAME_HEIGHT / 2))

# ----------------------------------------------------------------------------
# 4. Seat cushions — two, with a small gap so the seam stays visible
# ----------------------------------------------------------------------------
SEAT_TOP_OF_FRAME = FRAME_BOTTOM + FRAME_HEIGHT
SEAT_HEIGHT, SEAT_WIDTH, SEAT_DEPTH, CUSHION_GAP = 0.20, 0.90, 0.82, 0.03

seat_l = Box(SEAT_WIDTH, SEAT_DEPTH, SEAT_HEIGHT)
seat_l.rounding(0.055)
seat_l.move((SEAT_WIDTH / 2 + CUSHION_GAP / 2, -0.03,
             SEAT_TOP_OF_FRAME + SEAT_HEIGHT / 2 - 0.03))

seat_r = Box(SEAT_WIDTH, SEAT_DEPTH, SEAT_HEIGHT)
seat_r.rounding(0.055)
seat_r.move((-(SEAT_WIDTH / 2 + CUSHION_GAP / 2), -0.03,
             SEAT_TOP_OF_FRAME + SEAT_HEIGHT / 2 - 0.03))

# ----------------------------------------------------------------------------
# 5. Backrest cushions — two, tilted back a few degrees
# ----------------------------------------------------------------------------
BACK_WIDTH, BACK_DEPTH, BACK_HEIGHT, BACK_TILT = 0.90, 0.22, 0.48, np.deg2rad(9)

back_l = Box(BACK_WIDTH, BACK_DEPTH, BACK_HEIGHT)
back_l.rounding(0.06)
back_l.move((BACK_WIDTH / 2 + CUSHION_GAP / 2, -0.38,
             SEAT_TOP_OF_FRAME + BACK_HEIGHT / 2 + 0.02))
back_l.rotate(BACK_TILT, (1, 0, 0))

back_r = Box(BACK_WIDTH, BACK_DEPTH, BACK_HEIGHT)
back_r.rounding(0.06)
back_r.move((-(BACK_WIDTH / 2 + CUSHION_GAP / 2), -0.38,
             SEAT_TOP_OF_FRAME + BACK_HEIGHT / 2 + 0.02))
back_r.rotate(BACK_TILT, (1, 0, 0))

# ----------------------------------------------------------------------------
# 6. Arms — flat-top track arms, smoothly blended into the frame
# ----------------------------------------------------------------------------
ARM_HEIGHT = 0.56
arm_l = Box(0.18, 0.94, ARM_HEIGHT)
arm_l.rounding(0.04)
arm_l.move((0.955, -0.02, FRAME_BOTTOM + ARM_HEIGHT / 2))

arm_r = Box(0.18, 0.94, ARM_HEIGHT)
arm_r.rounding(0.04)
arm_r.move((-0.955, -0.02, FRAME_BOTTOM + ARM_HEIGHT / 2))

# ----------------------------------------------------------------------------
# 7. Assemble the full couch
# ----------------------------------------------------------------------------
body = UNION.combine(frame, *legs)
body = smooth(body, arm_l, 0.05)
body = smooth(body, arm_r, 0.05)
couch = UNION.combine(body, seat_l, seat_r, back_l, back_r)

sdf = couch.create(coor)
print(f"Grid: {coor.shape[1]:,} points, resolution {co_res}")
print(f"Inside fraction: {hard_binarization(sdf, 0).mean():.3f}")

# ----------------------------------------------------------------------------
# 8. Cross-section figures
# ----------------------------------------------------------------------------
binary_3d = smarter_reshape(hard_binarization(sdf, 0), co_res)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(binary_3d[:, co_res[1] // 2, :].T, cmap="Blues", origin="lower",
               extent=(-CO_SIZE[0] / 2, CO_SIZE[0] / 2, -CO_SIZE[2] / 2, CO_SIZE[2] / 2))
axes[0].set_title("Side view (XZ)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("z")

axes[1].imshow(binary_3d[co_res[0] // 2, :, :].T, cmap="Purples", origin="lower",
               extent=(-CO_SIZE[1] / 2, CO_SIZE[1] / 2, -CO_SIZE[2] / 2, CO_SIZE[2] / 2))
axes[1].set_title("Front view (YZ)")
axes[1].set_xlabel("y"); axes[1].set_ylabel("z")

axes[2].imshow(binary_3d[:, :, co_res[2] // 2].T, cmap="Greens", origin="lower",
               extent=(-CO_SIZE[0] / 2, CO_SIZE[0] / 2, -CO_SIZE[1] / 2, CO_SIZE[1] / 2))
axes[2].set_title("Top view (XY)")
axes[2].set_xlabel("x"); axes[2].set_ylabel("y")

plt.tight_layout()
plt.savefig("couch_cross_sections.png", dpi=140)
plt.close(fig)
print("Saved couch_cross_sections.png")

# ----------------------------------------------------------------------------
# 9. Surface point cloud
# ----------------------------------------------------------------------------
dx = CO_SIZE[0] / co_res[0]
surface_mask = np.abs(sdf) < dx * 1.2
surface_pts = coor[:, surface_mask].T  # (N, 3)

np.save("couch_surface_points.npy", surface_pts)
np.savetxt("couch_surface_points.xyz", surface_pts, fmt="%.5f")
print(f"Surface point cloud: {surface_pts.shape[0]:,} points "
      f"-> couch_surface_points.npy / .xyz")

# ----------------------------------------------------------------------------
# 10. Colour-code the point cloud by which part it belongs to
# ----------------------------------------------------------------------------
# Evaluate each individual part's own SDF at the surface points and assign
# each point to whichever part it sits deepest inside of (most negative SDF).
parts = {
    "legs": (UNION.combine(*legs), "#6b4a33"),   # walnut brown
    "frame": (frame, "#3f3f3f"),                  # dark grey
    "seat": (UNION.combine(seat_l, seat_r), "#d8c9a3"),   # linen beige
    "back": (UNION.combine(back_l, back_r), "#cbb98d"),   # slightly deeper beige
    "arms": (UNION.combine(arm_l, arm_r), "#8a8578"),      # warm grey upholstery
}

probe = surface_pts.T  # (3, N)
part_names = list(parts.keys())
distances = np.stack([parts[name][0].create(probe) for name in part_names], axis=0)
labels = np.argmin(distances, axis=0)
colors = np.array([parts[name][1] for name in part_names])[labels]

# ----------------------------------------------------------------------------
# 11. 3D scatter figures
# ----------------------------------------------------------------------------
fig = plt.figure(figsize=(14, 7))

ax1 = fig.add_subplot(121, projection="3d")
ax1.scatter(surface_pts[:, 0], surface_pts[:, 1], surface_pts[:, 2],
            c=colors, s=1.2, linewidths=0)
ax1.set_box_aspect(CO_SIZE)
ax1.view_init(elev=20, azim=-60)
ax1.set_title("Three-quarter view")
ax1.set_axis_off()

ax2 = fig.add_subplot(122, projection="3d")
ax2.scatter(surface_pts[:, 0], surface_pts[:, 1], surface_pts[:, 2],
            c=colors, s=1.2, linewidths=0)
ax2.set_box_aspect(CO_SIZE)
ax2.view_init(elev=12, azim=-100)
ax2.set_title("Front-side view")
ax2.set_axis_off()

plt.tight_layout()
plt.savefig("couch_pointcloud_3d.png", dpi=140)
plt.close(fig)
print("Saved couch_pointcloud_3d.png")

print("\nDone.")

Grid: 1,400,035 points, resolution (181, 91, 85)
Inside fraction: 0.541
Saved couch_cross_sections.png
Surface point cloud: 116,047 points -> couch_surface_points.npy / .xyz
Saved couch_pointcloud_3d.png

Done.


## Outcome

***The entire output is too large to show here, but the code runs successfully and does generate a modern couch. The presentation of the final object could be better.***